In [181]:
from pandas.core.frame import DataFrame
import pandas as pd
import os

symbol = "VNM"
daily_file = f"processed_stck_data/{symbol}_RRV.csv"
df_daily: DataFrame = pd.read_csv(daily_file, parse_dates=["Date"])

In [182]:
from vnstock3 import Vnstock
stock = Vnstock().stock(symbol=symbol, source='TCBS')

df_quote_history= stock.quote.history(start='2025-01-01', end='2025-01-31')
df_cash_flow = stock.finance.cash_flow(period='quarter') 
df_income_statement = stock.finance.income_statement(period='quarter')
df_ratio = stock.finance.ratio(period='quarter')

df_quote_history.to_csv(f'files/{symbol}_2025_quote_history.csv', index=False)
df_cash_flow.to_csv(f'files/{symbol}_2025_cash_flow.csv', index=False)
df_income_statement.to_csv(f'files/{symbol}_2025_income_statement.csv', index=False)
df_ratio.to_csv(f'files/{symbol}_2025_ratio.csv', index=False)
print('done!')

done!


In [183]:
start_date = pd.to_datetime("2013-03-31")
end_date   = pd.to_datetime("2024-12-31")

In [184]:
# Filter daily data to this date range
df_daily = df_daily[(df_daily["Date"] >= start_date) & (df_daily["Date"] <= end_date)]
df_daily.sort_values("Date", inplace=True)

In [185]:
df_daily.head()

,Date,Close,Open,High,Low,Volume,Change,Symbol,return_day,return_week,...,MFI14,MOM1,MOM3,MOM7,CCI12,CCI20,ROCR3,ROCR12,WILLR,TRIX
0,2018-01-01,172881.0,172472.0,173208.0,170510.0,447310.0,1.39,VNM_RRV,0.000000,-0.00246,...,47.679160,0.0,0.0,-50.0,-70.588235,-76.015727,1.000000,1.006203,-12.120089,-0.029148
1,2018-01-02,172881.0,172472.0,173208.0,170510.0,447310.0,1.39,VNM_RRV,0.000000,-0.00246,...,47.679160,0.0,0.0,-50.0,-70.588235,-76.015727,1.000000,1.006203,-12.120089,0.000000
2,2018-01-03,174842.0,174516.0,175742.0,173371.0,668840.0,1.13,VNM_RRV,0.011343,-0.00246,...,100.000000,1961.0,0.0,-50.0,100.000000,100.000000,1.000000,1.006203,-17.201835,0.002215
3,2018-01-04,175578.0,174189.0,175742.0,174189.0,845220.0,0.42,VNM_RRV,0.004210,-0.00246,...,100.000000,736.0,2697.0,-50.0,79.404894,79.404894,1.015600,1.006203,-3.134557,0.006647
4,2018-01-05,174516.0,174924.0,175742.0,173698.0,622630.0,-0.60,VNM_RRV,-0.006049,-0.00246,...,70.894177,-1062.0,1635.0,-50.0,46.430738,46.430738,1.009457,1.006203,-23.432722,0.011159


In [186]:
df_daily.tail()

,Date,Close,Open,High,Low,Volume,Change,Symbol,return_day,return_week,...,MFI14,MOM1,MOM3,MOM7,CCI12,CCI20,ROCR3,ROCR12,WILLR,TRIX
2552,2024-12-27,63800.000000,63900.0,64100.000000,63700.000000,2.440000e+06,0.00,VNM_RRV,0.000000,-0.004656,...,45.056392,0.000000,-0.800000,-695.300000,-78.934513,-33.906198,0.999987,0.993294,-69.292193,0.001897
2553,2024-12-28,63633.333333,63800.0,63966.666667,63566.666667,2.123333e+06,-0.26,VNM_RRV,-0.002612,-0.004173,...,38.512098,-166.666667,-266.666667,-663.533333,-109.630420,-83.616803,0.995827,0.986635,-82.213091,0.000756
2554,2024-12-29,63466.666667,63700.0,63833.333333,63433.333333,1.806667e+06,-0.52,VNM_RRV,-0.002619,-0.005237,...,31.705272,-166.666667,-333.333333,-631.766667,-130.977899,-127.658529,0.994775,0.990144,-95.133990,-0.001426
2555,2024-12-30,63300.000000,63600.0,63700.000000,63300.000000,1.490000e+06,-0.78,VNM_RRV,-0.002626,-0.009390,...,24.142446,-166.666667,-500.000000,-600.000000,-148.658138,-168.322485,0.992163,0.986017,-100.000000,-0.004718
2556,2024-12-31,63400.000000,63400.0,63800.000000,63300.000000,1.640000e+06,0.16,VNM_RRV,0.001580,-0.006270,...,29.531488,100.000000,-233.333333,-400.800000,-118.409611,-134.581838,0.996333,0.999938,-92.275606,-0.008297


In [187]:
df_daily.isna().sum()[df_daily.isna().sum() > 0]

Series([], dtype: int64)

In [188]:
# Mapping for quarter end dates
quarter_end_map = {1: '-03-31', 2: '-06-30', 3: '-09-30', 4: '-12-31'}

def load_and_process_quarterly(file_path, start_date, end_date):
    df = pd.read_csv(file_path)

    if not {"year", "quarter"}.issubset(df.columns):
        raise ValueError(f"File {file_path} must contain 'year' and 'quarter' columns.")

    df["Quarter_End"] = pd.to_datetime(
        df["year"].astype(str) + df["quarter"].astype(int).map(quarter_end_map)
    )
    
    df = df[(df["Quarter_End"] >= start_date) & (df["Quarter_End"] <= end_date)]
    
    df.sort_values("Quarter_End", inplace=True)
    return df

In [189]:
cash_flow_file = f"files/{symbol}_2025_cash_flow.csv"
financial_reports_file = f"files/{symbol}_2025_income_statement.csv"
stock_ratio_file = f"files/{symbol}_2025_ratio.csv"

start_date = pd.to_datetime("2013-03-31")
end_date   = pd.to_datetime("2024-12-31")

df_cash_flow = load_and_process_quarterly(cash_flow_file, start_date, end_date)
df_financial  = load_and_process_quarterly(financial_reports_file, start_date, end_date)
df_stock_ratio = load_and_process_quarterly(stock_ratio_file, start_date, end_date)

In [190]:
df_merged = pd.merge_asof(
    df_daily.sort_values("Date"),
    df_cash_flow,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_cf")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_financial,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_fr")
)

df_merged = pd.merge_asof(
    df_merged.sort_values("Date"),
    df_stock_ratio,
    left_on="Date",
    right_on="Quarter_End",
    direction="backward",
    suffixes=("", "_sr")
)

In [191]:
len(df_merged.columns.tolist())

128

In [192]:
df_merged.columns.tolist()

['Date',
 'Close',
 'Open',
 'High',
 'Low',
 'Volume',
 'Change',
 'Symbol',
 'return_day',
 'return_week',
 'return_month',
 'volatility_day',
 'volatility_week',
 'volatility_month',
 'liquidity_day',
 'liquidity_week',
 'liquidity_month',
 'high_minus_close',
 'low_minus_open',
 'cumulative_return',
 'Stochastic_Osc',
 'ATR',
 'SMA_3',
 'SMA_7',
 'SMA_14',
 'SMA_21',
 'SMA_50',
 'SMA_100',
 'WMA_3',
 'WMA_7',
 'WMA_14',
 'WMA_21',
 'WMA_50',
 'WMA_100',
 'EMA6',
 'EMA12',
 'EMA26',
 'outMACD',
 'outMACDSignal',
 'outMACDHist',
 'RSI6',
 'RSI12',
 'RSI14',
 'StochRSI_6',
 'StochRSI_12',
 'StochRSI_14',
 'BBANDSMIDDLE',
 'BBANDSUPPER',
 'BBANDSLOWER',
 'OBV',
 'MFI14',
 'MOM1',
 'MOM3',
 'MOM7',
 'CCI12',
 'CCI20',
 'ROCR3',
 'ROCR12',
 'WILLR',
 'TRIX',
 'quarter',
 'year',
 'invest_cost',
 'from_invest',
 'from_financial',
 'from_sale',
 'free_cash_flow',
 'Quarter_End',
 'quarter_fr',
 'year_fr',
 'revenue',
 'year_revenue_growth',
 'quarter_revenue_growth',
 'cost_of_good_sold',


In [193]:
cols_to_drop = ['Quarter_End_sr', 'year_sr', 'quarter_sr', 'Quarter_End_fr', 'year_fr', 'quarter_fr', 'Quarter_End', 'year', 'quarter' ]
df_merged.drop(columns=cols_to_drop, inplace=True)

In [194]:
# Step 1: Create a mapping of old → new names
original_columns = [
    'Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change', 'Symbol',
    'return_day', 'return_week', 'return_month', 'volatility_day', 'volatility_week', 'volatility_month',
    'liquidity_day', 'liquidity_week', 'liquidity_month', 'high_minus_close', 'low_minus_open',
    'cumulative_return', 'Stochastic_Osc', 'ATR', 'SMA_3', 'SMA_7', 'SMA_14', 'SMA_21', 'SMA_50', 'SMA_100',
    'WMA_3', 'WMA_7', 'WMA_14', 'WMA_21', 'WMA_50', 'WMA_100', 'EMA6', 'EMA12', 'EMA26',
    'outMACD', 'outMACDSignal', 'outMACDHist', 'RSI6', 'RSI12', 'RSI14',
    'StochRSI_6', 'StochRSI_12', 'StochRSI_14',
    'BBANDSMIDDLE', 'BBANDSUPPER', 'BBANDSLOWER',
    'OBV', 'MFI14', 'MOM1', 'MOM3', 'MOM7', 'CCI12', 'CCI20', 'ROCR3', 'ROCR12', 'WILLR', 'TRIX'
]

renamed_columns = [
    'date', 'close', 'open', 'high', 'low', 'volume', 'change', 'symbol',
    'return_day', 'return_week', 'return_month', 'volatility_day', 'volatility_week', 'volatility_month',
    'liquidity_day', 'liquidity_week', 'liquidity_month', 'high_minus_close', 'low_minus_open',
    'cumulative_return', 'stochastic_osc', 'atr', 'sma_3', 'sma_7', 'sma_14', 'sma_21', 'sma_50', 'sma_100',
    'wma_3', 'wma_7', 'wma_14', 'wma_21', 'wma_50', 'wma_100', 'ema_6', 'ema_12', 'ema_26',
    'out_macd', 'out_macd_signal', 'out_macd_hist', 'rsi_6', 'rsi_12', 'rsi_14',
    'stochrsi_6', 'stochrsi_12', 'stochrsi_14',
    'bbands_middle', 'bbands_upper', 'bbands_lower',
    'obv', 'mfi_14', 'mom_1', 'mom_3', 'mom_7', 'cci_12', 'cci_20', 'rocr_3', 'rocr_12', 'willr', 'trix'
]

rename_map = dict(zip(original_columns, renamed_columns))
df_merged.rename(columns=rename_map, inplace=True)

In [195]:
# ======== Step 5: Save the combined DataFrame ========
output_file = f"feature_engineered/feature_engineered_{symbol}_RRV.csv"
df_merged.to_csv(output_file, index=False)
print(f"Saved combined feature-engineered file to {output_file}")

Saved combined feature-engineered file to feature_engineered/feature_engineered_VNM_RRV.csv


# The End